Lab 03 - Escrow & payment channels (Module 3).

Open a pre-funded payment channel, invoke through hub-lite, then close and
inspect the refund — the micropayment pattern behind Protocol v2 escrow.

Run:  python labs/lab03_escrow_channel.py   (COURSE_LANG=ru|es to localize)
Needs: pip install -e ".[hub-lite]"

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/alexar76/aimarket-courses/blob/main/agent-economy-course/notebooks/{})


In [ ]:
# Setup — run this cell once per session
# Hub-lite (~5 MB) — fast install for hub/escrow labs
import os
import subprocess
import sys

REPO = "https://github.com/alexar76/aimarket-courses.git"
DEST = "/content/aimarket-courses"

if not os.path.isdir(DEST):
    subprocess.run(["git", "clone", "-q", "--depth", "1", REPO, DEST], check=True)
WORKDIR = os.path.join(DEST, "agent-economy-course")
os.chdir(WORKDIR)

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", ".[hub-lite]"], check=True)
os.environ.setdefault("COURSE_LANG", "en")  # change to ru or es


In [ ]:
"""Lab 03 - Escrow & payment channels (Module 3).

Open a pre-funded payment channel, invoke through hub-lite, then close and
inspect the refund — the micropayment pattern behind Protocol v2 escrow.

Run:  python labs/lab03_escrow_channel.py   (COURSE_LANG=ru|es to localize)
Needs: pip install -e ".[hub-lite]"
"""



from courselib.hub_lite import embedded_hub_lite
from courselib.i18n import get_translator
from courselib.orchestration import Trace


def main() -> None:
    t = get_translator()
    print(f"== {t('modules.m3.title')} ==")
    print(t("modules.m3.concept"))

    trace = Trace()

    with embedded_hub_lite() as hub:
        opened = hub.open_channel(budget_usd=0.25)
        channel_id = opened.get("channel_id", "")
        trace.emit("channel_open", channel_id=channel_id, budget=opened.get("budget_usd"))
        print(f"{t('ui.channel')}     :", channel_id or t("ui.none"))
        print(f"{t('labs.lab03.budget')}   : ${opened.get('budget_usd')}")

        out = hub.invoke("prod-summarize", "summarize@v1", {"text": t("labs.lab03.sample")})
        trace.emit("invoke", capability="summarize@v1", price_usd=out.get("price_usd"))
        print(f"\n{t('ui.invoke')} summarize@v1  {t('ui.price')}=${out.get('price_usd')}")

        closed = hub.close_channel(channel_id)
        trace.emit("channel_close", spent=closed.get("spent_usd"), refund=closed.get("refund_usd"))
        print(f"\n{t('labs.lab03.spent')}    : ${closed.get('spent_usd', 0)}")
        print(f"{t('labs.lab03.refund')}   : ${closed.get('refund_usd', 0)}")

    print(f"\n{t('ui.trace')} ({len(trace)} {t('ui.events')}):")
    for e in trace.events:
        print(" ", e["kind"], {k: v for k, v in e.items() if k not in ("t", "kind")})

    print(f"\n--- {t('exercises.heading')} ---")
    print(t("exercises.lab03_hint"))

main()
